In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [4]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="RobotsMali/afvoices",
    repo_type="dataset", local_dir="./afvoices", allow_patterns="human-corrected/*.parquet")

Fetching 130 files: 100%|██████████| 130/130 [00:35<00:00,  3.70it/s]


'/home/ubuntu/afvoices'

In [5]:
files = glob('afvoices/*/*.parquet')
len(files)

130

In [6]:
df = pd.read_parquet(files[0])
df

,text,duration,audio,label-v1,label-v2
0,Ni mɔgɔ dɔw yɛrɛ b'a fɔ[?],1.723991,{'bytes': b'RIFF8\xa4\x04\x00WAVEfmt \x10\x00\...,Ni mɔgɔ dɔ yɛrɛ b'a fɔ,Ni mɔgɔ dɔw yɛrɛ b'a fɔ
1,An ka tabiya bi taa sira kelen fɛ [?],1.788005,{'bytes': b'RIFFLh\x02\x00WAVEfmt \x10\x00\x00...,An ka tabiya bɛ taa sira kelen fɛ,An ka tabiya bɛ taa sira kelen fɛ
2,Aw ka laɲinin tɛ min ye,1.212000,{'bytes': b'RIFF\xda\xa1\x01\x00WAVEfmt \x10\x...,Aw ka laɲiniini tɛ min ye,Aw ka laɲini tɛ min ye
3,[cs] ne ka zonwaniw bɛ dumuni kɛ sufɛ o b'a kɛ...,2.364014,{'bytes': b'RIFF\xc0.\x03\x00WAVEfmt \x10\x00\...,Bon ne ka zani ni dumuniini ye su fɛ b waaa kɛ...,[cs] ne ka zonzani bɛ dumuni kɛ sufɛ u b'a kɛ ...
4,Lafiya kun bɛ ka kɛ [cs] a donna ji la k'a ko [?],2.620000,{'bytes': b'RIFF\xa0\r\x07\x00WAVEfmt \x10\x00...,Lafiya tun bɛ ka kɛ mɛ dipi'a donna ji la wa ko,Lafiya kun bɛ ka kɛ mɛtiri a donna ji la k'a ko
...,...,...,...,...,...
2005,A dɔw cɛ bɛ k'a fɔ ko ale fura fɛn fɛn san,2.171995,{'bytes': b'RIFF\x98\xec\x02\x00WAVEfmt \x10\x...,Ka don co ko ko ale furafɛfɛ san,A dɔw tun bɛ ka fɔ ko ale ye fura fɛn fɛn san[?]
2006,Kalan ko dɛmɛtɔ de ye [?],1.660000,{'bytes': b'RIFF\x10<\x02\x00WAVEfmt \x10\x00\...,Kalan ko dɛmɛtɔ de ye,Kalanko dɛmɛtɔ de[?]
2007,O yiri bɛ yaala,1.148000,{'bytes': b'RIFF\xcc\x8b\x01\x00WAVEfmt \x10\x...,O yiri bɛ yaala,O ni yiriw bɛ yaala
2008,An b'a ɲɛfɔ dala fɔli de ye dɔnku ni dɔ na na ...,4.411995,{'bytes': b'RIFFX\xf0\x05\x00WAVEfmt \x10\x00\...,An b'a ɲɛfɔda fɔ olu de ye dɔn naani fɛ yen n'...,An b'a ɲɛfɔ dala fɔliw de ye[cs] ni dɔ nana i ...


In [8]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['label-v1'].iloc[i].strip()
            if len(t) < 5:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [22]:
data = multiprocessing(files, loop, cores = 5)

In [10]:
with open('afvoices.json', 'w') as fopen:
    json.dump(data, fopen)

In [11]:
audio_files = [d['audio_filename'] for d in data]

with open('afvoices-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [15]:
# !zip -rq afvoices_audio.zip afvoices_audio

In [16]:
# !hf upload malaysia-ai/Multilingual-TTS afvoices_audio.zip --repo-type=dataset

In [19]:
# !zip -rq afvoices_audio_neucodec.zip afvoices_audio_neucodec

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS afvoices_audio_neucodec.zip --repo-type=dataset

In [21]:
# !python3 embedding.py --file 'afvoices.json'

In [23]:
import json

with open('afvoices.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 258966/258966 [00:00<00:00, 2748895.64it/s]


258966

In [24]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'afvoices_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 258966/258966 [04:33<00:00, 946.82it/s] 


In [25]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [26]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'afvoices_audio/afvoices-human-corrected-train-00117-of-00126_0.mp3',
 'text': "Ni mɔgɔ dɔ yɛrɛ b'a fɔ",
 'speaker': 'afvoices_audio_0'}

In [27]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'afvoices')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 13.38ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  94%|█████████▎| 8.70MB / 9.29MB, 21.8MB/s  
Processing Files (1 / 1): 100%|██████████| 9.29MB / 9.29MB, 17.3MB/s  
Processing Files (1 / 1): 100%|██████████| 9.29MB / 9.29MB, 15.5MB/s  
New Data Upload: 100%|██████████| 9.29MB / 9.29MB, 15.5MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.06s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/00ae64df78567b935414a20fd1858e4b0d02a0d6', commit_message='Upload dataset', commit_description='', oid='00ae64df78567b935414a20fd1858e4b0d02a0d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)